<a href="https://colab.research.google.com/github/aymanr8-dot/ds2002-fa26/blob/main/notebooks/04-json-apis/2026_10_09_public_api_to_clean_table_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [17]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    last_problem = None
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.ConnectionError, requests.Timeout) as e:
            # network hiccup: worth retrying
            last_problem = repr(e)
        else:
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            if r.status_code not in RETRY_STATUSES:
                # 400 / 404 etc. -- our bug, retrying will not fix it
                r.raise_for_status()
                raise requests.HTTPError(f'unexpected status {r.status_code}', response=r)
            last_problem = f'HTTP {r.status_code}'
        if attempt < tries:
            time.sleep(2 ** attempt)   # 2s, 4s, 8s, ...
    raise RuntimeError(f'get_weather failed after {tries} attempts (last: {last_problem})')

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [18]:
START, END = '2026-09-05', '2026-10-03'
DATES = [START, END]
LAT, LON = 38.03, -78.51

used_fallback = False
picked = None
pulled_at = pd.Timestamp.now(tz='America/New_York')

try:
    raw = get_weather(START, END, lat=LAT, lon=LON)   # one request for the whole range
    picked = raw[raw['time'].isin(DATES)].reset_index(drop=True)
    print(f'pulled {len(raw)} days, kept {len(picked)}')
except (RuntimeError, requests.ConnectionError, requests.Timeout) as e:
    # only a genuine outage lands here; a 400/404 still raises so we see our own bug
    print(f'API call failed, will use fallback: {e}')

picked

pulled 29 days, kept 2


,time,temperature_2m_max,precipitation_sum
0,2026-09-05,29.4,2.5
1,2026-10-03,22.0,14.7


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [19]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if picked is None:
    used_fallback = True
    wx = FALLBACK.copy()
else:
    wx = picked.copy()

wx['source'] = 'fallback sample' if used_fallback else 'open-meteo'
print('used_fallback =', used_fallback)
wx

used_fallback = False


,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [20]:
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
wx['date'] = pd.to_datetime(wx['date'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']].sort_values('date').reset_index(drop=True)
wx.dtypes

,0
date,datetime64[ns]
temp_max,float64
precip_mm,float64
source,object


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [21]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [22]:
ranked = wx.sort_values('precip_mm', ascending=False).reset_index(drop=True)
wet, dry = ranked.iloc[0], ranked.iloc[1]
diff = wet['precip_mm'] - dry['precip_mm']
note = '' if not used_fallback else ' (NOTE: fallback sample values, not real observations)'

if diff == 0:
    print(f"Both {wet['date']:%A, %B %-d} and {dry['date']:%A, %B %-d} had the same rainfall "
          f"({wet['precip_mm']:.1f} mm){note}.")
else:
    print(f"{wet['date']:%A, %B %-d} was the wetter home weekend: {wet['precip_mm']:.1f} mm of rain "
          f"versus {dry['precip_mm']:.1f} mm on {dry['date']:%A, %B %-d}, "
          f"a difference of {diff:.1f} mm{note}.")

Saturday, October 3 was the wetter home weekend: 14.7 mm of rain versus 2.5 mm on Saturday, September 5, a difference of 12.2 mm.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [23]:
print('Source:      Open-Meteo Historical Weather API')
print(f'Endpoint:    {BASE}')
print(f'Variables:   temperature_2m_max (°C), precipitation_sum (mm), daily, America/New_York')
print(f'Coordinates: lat {LAT}, lon {LON} (Charlottesville, VA)')
print(f'Date range:  {START} to {END} (kept {", ".join(DATES)})')
print(f'Pulled at:   {pulled_at:%Y-%m-%d %H:%M %Z}')
print(f'Data status: {"FALLBACK SAMPLE -- not real observations" if used_fallback else "real data from open-meteo"}')

Source:      Open-Meteo Historical Weather API
Endpoint:    https://archive-api.open-meteo.com/v1/archive
Variables:   temperature_2m_max (°C), precipitation_sum (mm), daily, America/New_York
Coordinates: lat 38.03, lon -78.51 (Charlottesville, VA)
Date range:  2026-09-05 to 2026-10-03 (kept 2026-09-05, 2026-10-03)
Pulled at:   2026-10-11 15:59 EDT
Data status: real data from open-meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame instead of raising, nothing would fail downstream. A left merge onto 200 days of sales keeps all 200 rows, but every instance of precip_nm would be NaN. Therefore, the rain-vs-revenue chart would come out as empty or show everyday at 0 mm if the NaNs get filled. As there are no cell errors, row count looks right and NaN reads as ordinary missing data, nobody would catch it. The failure in this scenario shows up as a finding rather than a traceback.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [24]:
CITIES = {
    'Charlottesville': (38.03, -78.51),
    'Orlando': (28.54, -81.38),
}

# sample values per city so this cell still runs without a network
FALLBACK_BY_CITY = {
    'Charlottesville': FALLBACK,
    'Orlando': pd.DataFrame({
        'time': ['2026-09-05', '2026-10-03'],
        'temperature_2m_max': [32.8, 30.5],
        'precipitation_sum': [6.2, 0.0],
    }),
}

frames = []
fallback_cities = []
for city, (lat, lon) in CITIES.items():
    try:
        df = get_weather(START, END, lat=lat, lon=lon)
        df = df[df['time'].isin(DATES)].copy()
        df['source'] = 'open-meteo'
    except (RuntimeError, requests.ConnectionError, requests.Timeout) as e:
        # genuine outage only; a 400/404 still raises so we see our own bug
        print(f'{city}: API call failed, using fallback: {e}')
        df = FALLBACK_BY_CITY[city].copy()
        df['source'] = 'fallback sample'
        fallback_cities.append(city)
    df['city'] = city
    frames.append(df)
    time.sleep(1)  # be nice to the free API

wx_long = (
    pd.concat(frames, ignore_index=True)
      .rename(columns={'time': 'date',
                       'temperature_2m_max': 'temp_max',
                       'precipitation_sum': 'precip_mm'})
)
wx_long['date'] = pd.to_datetime(wx_long['date'])
wx_long = wx_long[['city', 'date', 'temp_max', 'precip_mm', 'source']]

assert len(wx_long) == len(CITIES) * len(DATES), f'expected 4 rows, got {len(wx_long)}'
assert pd.api.types.is_datetime64_any_dtype(wx_long['date'])
assert wx_long['precip_mm'].notna().all(), 'missing precipitation'
assert (wx_long['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('fallback used for:', fallback_cities if fallback_cities else 'no cities (all real data)')
wx_long

fallback used for: no cities (all real data)


,city,date,temp_max,precip_mm,source
0,Charlottesville,2026-09-05,29.4,2.5,open-meteo
1,Charlottesville,2026-10-03,22.0,14.7,open-meteo
2,Orlando,2026-09-05,32.1,3.9,open-meteo
3,Orlando,2026-10-03,30.4,14.5,open-meteo
